In [2]:
import gradio as gr

- Create models first

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("/home/ophelia/Desktop/University/2026/Datathon/UP_DIRISA_2026/data/processed/train_features.csv")
#y = target

In [4]:
label_encoder = LabelEncoder()

for col in df.select_dtypes(include=['object']).columns:
    df[col] = label_encoder.fit_transform(df[col])


/tmp/ipykernel_9808/222651402.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [5]:
#x is input, y is target
y = df[['Target_Turnout']].values
x = df.drop(['Target_Turnout'], axis=1).values


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42)


In [7]:
model = xgb.XGBRegressor(objective='reg:squarederror',
                         n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

#R2 = 0.934
#print(f'RMSE: {rmse:.3f}')
print(f'R²: {r2:.3f}')

R²: 0.570


In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

y_pred = grid_search.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'R²: {r2:.3f}')

#R2 = 0.941

Fitting 3 folds for each of 36 candidates, totalling 108 fits


R²: 0.624


In [10]:
#save model
import joblib

saved_model = joblib.dump(grid_search, "/home/ophelia/Desktop/University/2026/Datathon/UP_DIRISA_2026/src/model.pkl")

- Create function to use models

In [138]:
def model_pred(input):

    for col in input.select_dtypes(include=['object']).columns:
        input[col] = label_encoder.fit_transform(input[col])

    output = grid_search.predict(input)

    output = pd.DataFrame(output)
    output.columns = ["Predicted Turnout"]

    return output


- gr block ui code

In [139]:
with gr.Blocks() as demo:
    label = gr.Label("Enter input data row below")
    example = [
        "Province",
        "Ward",
        "MunicipalityCode",
        "RegisteredVoters_prior",
        "Turnout_prior",
        "IsMetro",
        "LogRegisteredVoters_prior",
        "MunicipalityAvgTurnout_prior",
        "SpoiltRatio_prior",
        "SnapshotYear"
    ]
    data_ex = np.array([["Eastern Cape","Ward 29200001","BUF",8851,0.5708959439611343,True,9.088285725968877,0.555891318766564,0.01583217890362161,2016]])
    input = gr.DataFrame(value=data_ex, headers=example, column_count=10, row_count=2)
    output = gr.DataFrame(headers = ["Predicted Turnout"])
    pred = gr.Button("Predict Turnout")
    pred.click(model_pred, inputs=input, outputs=output)

In [ ]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.


/tmp/ipykernel_10473/271754824.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in input.select_dtypes(include=['object']).columns:
/tmp/ipykernel_10473/271754824.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes